# ThirstyAI: reproduce the accounting and India context

Independent portfolio analysis. Observed/reported groundwater and illustrative scenarios are kept separate. Run `python scripts/reproduce.py` from the repository first. No optional notebook packages are required for the code cells.


In [1]:
from pathlib import Path
import sys, csv, sqlite3
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from thirstyai.model import Inputs, calculate, compare, break_even_grid
print('Project:', ROOT.name)


Project: thirstyai


## 1. Account for both pathways

For IT kWh E, PUE p, direct consumptive WUE w and grid factor i: total operational water = E(w + pi). Hypothetical inputs below are not facility measurements.


In [2]:
baseline = Inputs()
alternative = Inputs(pue=1.4, wue=0.1)
result = compare(baseline, alternative)
print('Baseline liters:', result['baseline']['total_l'])
print('Alternative liters:', result['alternative']['total_l'])
print('Additional facility MWh:', result['energy_change_mwh'])
print('Shared-grid break-even L/kWh:', round(break_even_grid(baseline, alternative), 2))


Baseline liters: 2200000.0
Alternative liters: 1500000.0
Additional facility MWh: 200.0
Shared-grid break-even L/kWh: 4.5


## 2. A lower direct water result can reverse at the system boundary

Change the shared grid intensity while preserving IT work. Negative savings means the alternative consumes more operational water under those assumptions.


In [3]:
for grid in [0, 1, 2, 4.5, 6]:
    r = compare(Inputs(grid_l_kwh=grid), Inputs(pue=1.4, wue=0.1, grid_l_kwh=grid))
    print(grid, round(r['water_saved_l']), 'L saved (conditional)')


0 900000 L saved (conditional)
1 700000 L saved (conditional)
2 500000 L saved (conditional)
4.5 0 L saved (conditional)
6 -300000 L saved (conditional)


## 3. Preserve published groundwater definitions

Source: CGWB/PIB assessment 2025, Annexure I. Annual all-use flows, not AI extraction. State ratios are not facility assessments. The query below identifies annual extraction above the assessed annual extractable resource without inventing causal or site-risk claims.


In [4]:
with sqlite3.connect(ROOT / 'outputs/thirstyai.sqlite') as connection:
    rows = connection.execute('SELECT state, stage_pct FROM groundwater WHERE stage_pct > 100 ORDER BY stage_pct DESC').fetchall()
    print(rows)
    print('State/UT coverage:', connection.execute('SELECT COUNT(*) FROM groundwater').fetchone()[0])


[('Punjab', 156.36), ('Rajasthan', 147.11), ('Haryana', 136.75)]
State/UT coverage: 36


## 4. Missingness and source rounding

Do not reinterpret a small rounded volume as missing, or recompute an official ratio from rounded inputs.


In [5]:
with (ROOT / 'data/processed/india_groundwater_2025.csv').open() as f:
    rows = list(csv.DictReader(f))
small = next(r for r in rows if r['state'] == 'Lakshadweep')
print({k:small[k] for k in ['state','extraction_bcm','extractable_bcm','stage_pct','source_id']})


{'state': 'Lakshadweep', 'extraction_bcm': '0.0', 'extractable_bcm': '0.01', 'stage_pct': '57.79', 'source_id': 'CGWB25'}


## Interpretation

The model is useful for checking assumptions and trade-offs. It does not establish a universal water-per-prompt number, an optimal city, a global AI water total, manufacturing water allocation or measured savings. See docs/methodology.md and research/claims.json.
